# Boundary Experiment — Account A
**P=4**, γ ∈ {0.0, 0.3, 0.6, 0.9}, C=21, ρ=0.5, seeds {42, 123, 456}, modes CI + CD

24 runs total. Output: `results_boundary_P4_acctA.csv`

In [ ]:
# ── Cell 1: Environment ──────────────────────────────────────────────────────
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_PATH = Path('/kaggle/working/results_boundary_P4_acctA.csv')

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Data generator ───────────────────────────────────────────────────
# Leader-follower VAR(1) process.
# Channels 0-9:  leaders  (pure AR(1), phi=0.8)
# Channels 10-19: followers (each follows leader i with coupling strength gamma)
# Channel 20:    isolate  (pure AR(1), internal negative control)
# Transition matrix A is lower-triangular → spectral radius = phi = 0.8 for all gamma.

N_LEADERS   = 10
N_TOTAL     = 21   # C
PHI         = 0.8  # AR coefficient
NOISE_STD   = 0.1
N_TIMESTEPS = 20_000


def generate(gamma: float, seed: int) -> np.ndarray:
    """Generate VAR(1) time series of shape (N_TIMESTEPS, N_TOTAL)."""
    rng = np.random.default_rng(seed)
    A = np.zeros((N_TOTAL, N_TOTAL))
    np.fill_diagonal(A, PHI)
    for i in range(N_LEADERS):
        A[i + N_LEADERS, i] = gamma  # follower i+10 depends on leader i

    X = np.zeros((N_TIMESTEPS, N_TOTAL))
    X[0] = rng.standard_normal(N_TOTAL) * NOISE_STD
    noise = rng.standard_normal((N_TIMESTEPS - 1, N_TOTAL)) * NOISE_STD
    for t in range(1, N_TIMESTEPS):
        X[t] = A @ X[t - 1] + noise[t - 1]
    return X


def split_and_normalise(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Chronological 70/10/20 split with train-statistics normalisation."""
    T = len(X)
    n_train = int(T * 0.70)
    n_val   = int(T * 0.10)

    train = X[:n_train]
    val   = X[n_train:n_train + n_val]
    test  = X[n_train + n_val:]

    mean = train.mean(axis=0, keepdims=True)
    std  = train.std(axis=0, keepdims=True) + 1e-8

    return (train - mean) / std, (val - mean) / std, (test - mean) / std

In [ ]:
# ── Cell 3: Experiment configuration ─────────────────────────────────────────
GAMMAS   = [0.0, 0.3, 0.6, 0.9]
SEEDS    = [42, 123, 456]
MODES    = ['CI', 'CD']

# Architecture — P=4 fixed for this account
LOOKBACK  = 512
PRED_LEN  = 96
PATCH_SIZE = 4
STRIDE     = PATCH_SIZE // 2   # = 2
N_PATCHES  = (LOOKBACK - PATCH_SIZE) // STRIDE + 1  # = 255
D_MODEL    = 64
N_HEADS    = 8
N_LAYERS   = 3
DROPOUT    = 0.2

# Training
BATCH_SIZES  = {'CI': 128, 'CD': 8}
LR           = 1e-4
WARMUP_EPOCHS = 10
MAX_EPOCHS    = 50
PATIENCE      = 10

print(f'P={PATCH_SIZE}  S={STRIDE}  N_patches={N_PATCHES}')
print(f'Head: Linear({N_PATCHES * D_MODEL}, {PRED_LEN})')
print(f'Total runs: {len(GAMMAS) * len(SEEDS) * len(MODES)}')

In [ ]:
# ── Cell 4: Model definitions ─────────────────────────────────────────────────
def extract_patches(x: torch.Tensor, patch_size: int, stride: int) -> torch.Tensor:
    """x: (B, L) → (B, N_patches, patch_size)"""
    return x.unfold(dimension=1, size=patch_size, step=stride)


class PatchEmbedding(nn.Module):
    def __init__(self, patch_size: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.proj    = nn.Linear(patch_size, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, N_patches, patch_size) → (B, N_patches, d_model)"""
        return self.dropout(self.proj(x))


class PatchTST_CI(nn.Module):
    """Channel-independent PatchTST. Each variate processed separately."""

    def __init__(
        self,
        lookback: int,
        pred_len: int,
        n_variates: int,
        patch_size: int,
        stride: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.stride     = stride
        self.n_patches  = (lookback - patch_size) // stride + 1
        self.n_variates = n_variates

        self.embed   = PatchEmbedding(patch_size, d_model, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head    = nn.Linear(self.n_patches * d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, L, C) → (B, pred_len, C)"""
        B, L, C = x.shape
        x_flat  = x.permute(0, 2, 1).reshape(B * C, L)
        patches = extract_patches(x_flat, self.patch_size, self.stride)
        emb     = self.embed(patches)
        enc     = self.encoder(emb)
        out     = self.head(enc.reshape(B * C, -1)).reshape(B, C, -1)
        return out.permute(0, 2, 1)


class PatchTST_CD(nn.Module):
    """Channel-dependent PatchTST. Cross-variate attention over all patches."""

    def __init__(
        self,
        lookback: int,
        pred_len: int,
        n_variates: int,
        patch_size: int,
        stride: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.stride     = stride
        self.n_patches  = (lookback - patch_size) // stride + 1
        self.n_variates = n_variates

        self.embed   = PatchEmbedding(patch_size, d_model, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        # Per-variate head: Linear(n_patches * d_model, pred_len)
        self.head = nn.Linear(self.n_patches * d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, L, C) → (B, pred_len, C)"""
        B, L, C = x.shape
        x_flat  = x.permute(0, 2, 1).reshape(B * C, L)
        patches = extract_patches(x_flat, self.patch_size, self.stride)
        # embed then restore (B, C, N_patches, d_model)
        emb = self.embed(patches).reshape(B, C, self.n_patches, -1)
        # flatten C and N_patches into one sequence for cross-variate attention
        seq = emb.reshape(B, C * self.n_patches, -1)
        enc = self.encoder(seq)
        # split back per variate and apply head
        enc_per_variate  = enc.reshape(B, C, self.n_patches, -1)
        flat_per_variate = enc_per_variate.reshape(B * C, -1)
        out = self.head(flat_per_variate).reshape(B, C, -1)
        return out.permute(0, 2, 1)


def build_model(mode: str) -> nn.Module:
    cls = PatchTST_CI if mode == 'CI' else PatchTST_CD
    return cls(
        LOOKBACK, PRED_LEN, N_TOTAL, PATCH_SIZE, STRIDE,
        D_MODEL, N_HEADS, N_LAYERS, DROPOUT
    ).to(DEVICE)


# ── Verification ──────────────────────────────────────────────────────────────
ci_head = build_model('CI').head
cd_head = build_model('CD').head
print(f'CI head: {ci_head}')  # must be Linear({N_PATCHES * D_MODEL}, {PRED_LEN})
print(f'CD head: {cd_head}')  # must be Linear({N_PATCHES * D_MODEL}, {PRED_LEN})
assert ci_head.in_features == N_PATCHES * D_MODEL, 'CI head mismatch'
assert cd_head.in_features == N_PATCHES * D_MODEL, 'CD head mismatch'

In [ ]:
# ── Cell 5: Dataset construction ──────────────────────────────────────────────
def make_windows(
    data: np.ndarray, lookback: int, pred_len: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Slide a window over data → (X, Y) tensors."""
    T, C = data.shape
    n_windows = T - lookback - pred_len + 1
    X = np.stack([data[i : i + lookback]           for i in range(n_windows)])
    Y = np.stack([data[i + lookback : i + lookback + pred_len] for i in range(n_windows)])
    return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)

In [ ]:
# ── Cell 6: Training functions ────────────────────────────────────────────────
def cosine_lr_with_warmup(
    optimizer: torch.optim.Optimizer,
    epoch: int,
    warmup_epochs: int,
    max_epochs: int,
    base_lr: float,
) -> None:
    """In-place LR update. Linear warmup then cosine decay to 1e-6."""
    min_lr = 1e-6
    if epoch < warmup_epochs:
        lr = base_lr * (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / max(1, max_epochs - warmup_epochs)
        lr = min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr


def train_one_run(
    mode: str,
    gamma: float,
    seed: int,
) -> dict:
    """Full training run. Returns result dict."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed(seed)

    # Data
    X_raw = generate(gamma, seed)
    train_data, val_data, test_data = split_and_normalise(X_raw)

    X_tr, Y_tr   = make_windows(train_data, LOOKBACK, PRED_LEN)
    X_val, Y_val = make_windows(val_data,   LOOKBACK, PRED_LEN)
    X_te, Y_te   = make_windows(test_data,  LOOKBACK, PRED_LEN)

    batch_size = BATCH_SIZES[mode]
    train_loader = DataLoader(
        TensorDataset(X_tr, Y_tr), batch_size=batch_size, shuffle=True, drop_last=True
    )
    val_loader = DataLoader(
        TensorDataset(X_val, Y_val), batch_size=batch_size * 4, shuffle=False
    )

    model     = build_model(mode)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scaler    = GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
    criterion = nn.MSELoss()

    best_val_loss   = float('inf')
    best_epoch      = 0
    best_total_steps = 0
    patience_counter = 0
    steps_so_far    = 0
    steps_per_epoch = len(train_loader)

    for epoch in range(MAX_EPOCHS):
        cosine_lr_with_warmup(optimizer, epoch, WARMUP_EPOCHS, MAX_EPOCHS, LR)

        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            with autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            steps_so_far += 1

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                with autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                    val_loss += criterion(model(xb), yb).item() * len(xb)
        val_loss /= len(val_loader.dataset)

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_epoch       = epoch + 1
            best_total_steps = steps_so_far
            patience_counter = 0
            torch.save(model.state_dict(), '/kaggle/working/best_model.pt')
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    # Evaluate on test set using best checkpoint
    model.load_state_dict(torch.load('/kaggle/working/best_model.pt', weights_only=True))
    model.eval()
    test_loader = DataLoader(
        TensorDataset(X_te, Y_te), batch_size=batch_size * 4, shuffle=False
    )
    test_mse = test_mae = 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                pred = model(xb)
            test_mse += nn.functional.mse_loss(pred, yb, reduction='sum').item()
            test_mae += nn.functional.l1_loss(pred, yb,  reduction='sum').item()
    n_test = len(test_loader.dataset) * PRED_LEN * N_TOTAL
    test_mse /= n_test
    test_mae /= n_test

    return {
        'dataset':          'leader_follower_var1',
        'C':                N_TOTAL,
        'rho':              0.5,
        'gamma':            gamma,
        'patch_size':       PATCH_SIZE,
        'mode':             mode,
        'seed':             seed,
        'test_mse':         round(test_mse, 6),
        'test_mae':         round(test_mae, 6),
        'best_epoch':       best_epoch,
        'batch_size':       batch_size,
        'steps_per_epoch':  steps_per_epoch,
        'total_steps':      best_total_steps,
    }

In [ ]:
# ── Cell 7: Main training loop ────────────────────────────────────────────────
# Build full experiment grid
EXPERIMENTS = [
    (gamma, mode, seed)
    for gamma in GAMMAS
    for mode  in MODES
    for seed  in SEEDS
]

# Resume: load completed runs from existing CSV
completed: set[tuple] = set()
if OUT_PATH.exists() and OUT_PATH.stat().st_size > 100:
    existing = pd.read_csv(OUT_PATH)
    for _, row in existing.iterrows():
        completed.add((row['gamma'], row['mode'], row['seed']))
    print(f'Resuming: {len(completed)} runs already complete.')
else:
    print('Starting fresh.')

remaining = [(g, m, s) for g, m, s in EXPERIMENTS if (g, m, s) not in completed]
total     = len(EXPERIMENTS)
done      = len(completed)
print(f'{len(remaining)} runs remaining of {total}.')

for gamma, mode, seed in remaining:
    done += 1
    label = f'[{done}/{total}] gamma={gamma} mode={mode} seed={seed}'
    print(f'{label} ...', flush=True)
    t0 = time.time()

    result = train_one_run(mode, gamma, seed)
    elapsed = int(time.time() - t0)

    print(
        f'{label} test_mse={result["test_mse"]:.4f}  '
        f'best_epoch={result["best_epoch"]}  ({elapsed}s)',
        flush=True,
    )

    # Append row and checkpoint immediately
    row_df = pd.DataFrame([result])
    write_header = not (OUT_PATH.exists() and OUT_PATH.stat().st_size > 100)
    row_df.to_csv(OUT_PATH, mode='a', header=write_header, index=False)

print(f'\nDone. Results saved to {OUT_PATH}')
print(f'Total runs: {total}')

In [ ]:
# ── Cell 8: Sanity check and summary ─────────────────────────────────────────
df = pd.read_csv(OUT_PATH)

print(f'Rows: {len(df)}  (expected {total})')
print(f'Patch size: {df["patch_size"].unique()}  (expected [{PATCH_SIZE}])')

print('\n=== Mean test MSE per (gamma, mode) ===')
summary = (
    df.groupby(['gamma', 'mode'])['test_mse']
    .agg(['mean', 'std'])
    .reset_index()
)
print(summary.to_string(index=False))

ci_mean = summary[summary['mode'] == 'CI'].set_index('gamma')['mean']
cd_mean = summary[summary['mode'] == 'CD'].set_index('gamma')['mean']

print('\n=== CD/CI ratio per gamma (>1.0 favours CI) ===')
for g in sorted(ci_mean.index):
    ratio = cd_mean[g] / ci_mean[g]
    print(f'  gamma={g}: CI={ci_mean[g]:.4f}  CD={cd_mean[g]:.4f}  ratio={ratio:.4f}')